# setup

In [43]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [44]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


# 문제 1. 데이터 탐색
판다스를 이용하여 데이터를 탐색하세요.

1. 전체 행 개수와 열 개수를 출력하세요.
2. 상위 5행을 출력하세요.
3. 각 열의 데이터 타입을 확인하세요.
4. `describe()`로 통계 정보를 확인하세요.
5. `target`의 값별 데이터 개수를 출력하세요.
6. 양성(1)과 음성(0)의 비율을 백분율로 출력하세요.

In [45]:
df.shape

(768, 9)

In [46]:
df.head()

,preg,plas,pres,skin,insu,mass,pedi,age,target
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [47]:
type(df["age"])

pandas.Series

In [48]:
df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age,target
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [49]:
x = df["target"].sum()
print("양성(1) 개수 : " , x)
print("음성(0) 개수 : " , df["target"].count()-x)

양성(1) 개수 :  268
음성(0) 개수 :  500


In [50]:
df["target"].mean()

np.float64(0.3489583333333333)

## 분석 질문

1. 전체 데이터에서 양성(1)의 비율은 몇 %인가요?
- 34.89%
2. 입력 특성 중 숫자 범위가 가장 큰 특성과 가장 작은 특성은 무엇인가요?
- insu(인슐린) , pedi(당뇨병 가족력 관련 지표)
3. 특성마다 숫자 범위가 다른 것이 KNN에 어떤 영향을 줄 수 있을까요?
- 특성별로 숫자 범위가 다르다는 것은 모델에 끼치는 영향력의 크기가 다르다는 의미임. 
- 그러면 숫자 범위가 큰 특성이 모델에 더 큰 영향을 주게 됨.
- 그래서 스케일링을 통해 모든 특성의 영향력을 균형있게 조정해야함.

# 문제 2. 결측치 탐색
데이터에서 누락된 측정값을 찾아보세요.

1. `isna().sum()`으로 결측치 개수를 출력하세요.
2. 준비 셀에서 안내한 5개 열의 0 개수를 각각 출력하세요.
3. 해당 5개 열에서 0을 `NaN`으로 변경하세요.
4. 다시 `isna().sum()`으로 결측치 개수를 확인하세요.
5. 결측치가 하나라도 포함된 행의 개수를 출력하세요.

**주의:** 이 단계에서는 결측치를 삭제하거나 다른 값으로 채우지 않습니다.

In [51]:
df.isna().sum()

preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64

In [52]:
zero_nan = ["plas" , "pres" , "skin" , "insu" , "mass"]
df[zero_nan].isna().sum()

plas    0
pres    0
skin    0
insu    0
mass    0
dtype: int64

In [53]:
import numpy as np
df[zero_nan] = df[zero_nan].replace(0, np.nan)

In [54]:
df.isna().sum()

preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64

In [55]:
# ???????????????????????????????????????????????
df.isna().any(axis=1).sum()

np.int64(376)

## 분석 질문

1. 처음 `isna()`를 실행했을 때 결측치가 발견되지 않은 이유는 무엇인가요?
-  이번 과제의 혈당, 혈압, 피부 두께, 인슐린, BMI 칼럼에서는 0을 누락된 측정값으로 간주하기 때문. 0은 숫자 데이터이지 결측치가 아님.
2. 모든 열의 0을 `NaN`으로 변경하면 안 되는 이유는 무엇인가요?
- 다른 임신 횟수나 나이 등에서의 0이라는 숫자는 정상적인 측정값이기 때문.

# 문제 3. 특성과 정답 분리 및 데이터 분할
머신러닝을 학습하기 위한 데이터를 준비하세요.

1. `target`을 제외한 열들을 `X`에 저장하세요.
2. `target`을 `y`에 저장하세요.
3. `X`와 `y`의 shape를 출력하세요.
4. `train_test_split()`으로 데이터를 분리하세요.

**분할 조건**

- 학습 데이터 80%
- 테스트 데이터 20%
- `random_state=42`
- `stratify=y`
1. `X_train`, `X_test`, `y_train`, `y_test`의 shape를 출력하세요.
2. 학습 데이터와 테스트 데이터의 양성·음성 개수를 각각 출력하세요.

In [56]:
X = df.drop(columns="target")
X.head()

,preg,plas,pres,skin,insu,mass,pedi,age
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33


In [57]:
y = df["target"]
y.head()

0    1
1    0
2    1
3    0
4    1
Name: target, dtype: int64

In [58]:
print("X : " , X.shape)
print("y : " , y.shape)

X :  (768, 8)
y :  (768,)


In [59]:
X_train , X_test , y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)
print("X : " , X_train.shape , X_test.shape)
print("y : " , y_train.shape , y_test.shape)

X :  (614, 8) (154, 8)
y :  (614,) (154,)


In [60]:
print("학습 데이터의 양성 개수: " , y_train.sum())
print("테스트 데이터의 양성 개수: " , y_test.sum())

학습 데이터의 양성 개수:  214
테스트 데이터의 양성 개수:  54


## 분석 질문

1. 학습 데이터와 테스트 데이터를 나누는 이유는 무엇인가요?
- 테스트는 학습되지 않은 새로운 데이터에 대해 정확하게 예측하는지 확인하는 과정임.
2. `stratify=y`를 사용하는 이유는 무엇인가요?
- y는 target(답안지) 데이터임. 그 안의 0(음성)과 1(양성)의 비율을 분할 데이터마다 동일하게 맞추는 과정임.
- 음성과 양성 데이터에 대해서 골고루 학습하고 테스트 할 수 있도록 비율을 맞추는 거임.
3. `random_state`를 지정하면 어떤 장점이 있나요?
- 난수 기준값을 고정시켜서 실행할 때마다 같은 데이터값을 받도록 함. 그러면 비교하기 쉬움.

# 문제 4. 결측치 처리

3번에서 분리한 학습 데이터와 테스트 데이터의 결측치를 처리하세요.

**조건**

- 결측치는 각 특성의 중앙값(`median`)으로 채우세요.
- 중앙값은 학습 데이터에서만 계산하세요.
- 테스트 데이터에는 학습 데이터에서 계산한 중앙값을 사용하세요.
- 원본 데이터는 유지하고 결측치를 처리한 데이터를 별도의 변수에 저장하세요.

다음 작업을 수행하세요.

1. 학습 데이터에서 각 특성의 중앙값을 계산하세요.
2. 학습 데이터의 결측치를 중앙값으로 채우세요.
3. 테스트 데이터의 결측치도 동일한 중앙값으로 채우세요.
4. 처리 후 학습 데이터와 테스트 데이터의 결측치 개수를 `isna().sum()`으로 출력하세요.
5. 처리 전후 데이터의 행 개수를 비교하세요.

In [61]:
mid = X_train.median()

In [62]:
X_train_cleaned = X_train.fillna(mid)

In [63]:
X_test_cleaned = X_test.fillna(mid)

In [64]:
print("학습 데이터 결측치\n" , X_train_cleaned.isna().sum())
print()
print("테스트 데이터 결측치\n" , X_test_cleaned.isna().sum())

학습 데이터 결측치
 preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64

테스트 데이터 결측치
 preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64


In [65]:
# 아니 처리 전후 데이터의 행 개수는 같겠지 아니야? 행 개수가 왜 달라지냐고. 걍 결측치에 값을 넣어줬을 뿐인데. 엥?
print("처리 전:" , X_train.shape , X_test.shape)
print("처리 후:" , X_train_cleaned.shape , X_test_cleaned.shape)

처리 전: (614, 8) (154, 8)
처리 후: (614, 8) (154, 8)


## 분석 질문

1. 결측치를 삭제하지 않고 중앙값으로 채우면 어떤 장점이 있나요?
- 학습데이터 누락없이 사용 가능. 중앙값으로 결측치를 대체하면 최솟값, 최댓값 어느 한쪽에 치우치지 않는 데이터가 되기 때문.
2. 테스트 데이터의 중앙값을 따로 계산해서 사용하면 안 되는 이유는 무엇인가요?
-  그러게요. 왜그랬대요? 이거는 잘 모르겟는데용? 학습데이터가 기반이 되는 데이터니까 그대로 써야하나?

# 문제 5. 첫 번째 KNN 학습

결측치 처리가 완료된 데이터를 이용하여 KNN을 학습하세요.

**조건**

- `KNeighborsClassifier` 사용
- `n_neighbors=5`
- 스케일링하지 않은 데이터 사용
- 4번에서 결측치를 처리한 학습·테스트 데이터 사용

다음 작업을 수행하세요.

1. KNN 객체를 생성하세요.
2. `fit()`으로 학습하세요.
3. 학습 정확도를 출력하세요.
4. 테스트 정확도를 출력하세요.
5. `predict()`로 테스트 데이터의 결과를 예측하세요.
6. 예측 결과와 실제 정답을 각각 앞 10개씩 출력하세요.

In [66]:
model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train_cleaned , y_train)

pred_train = model.predict(X_train_cleaned)
pred_test = model.predict(X_test_cleaned)

accuracy_train_1 = accuracy_score(y_train , pred_train)
accuracy_test_1 = accuracy_score(y_test , pred_test)

print("학습 정확도 :" , accuracy_train_1)
print("테스트 정확도 :" , accuracy_test_1)

recall_1 = recall_score(y_test , pred_test)

학습 정확도 :

 0.8013029315960912
테스트 정확도 : 0.6753246753246753


In [67]:
print("예측결과: " , pred_test[:10].tolist())
print("실제결과: " , y_test[:10].tolist())

예측결과:  [1, 0, 0, 0, 0, 0, 0, 1, 0, 1]
실제결과:  [0, 0, 0, 1, 0, 0, 1, 1, 0, 0]


## 분석 질문

1. KNN에서 k=5는 어떤 의미인가요?
- 가장 가까운 지표데이터 5개.
2. 학습 정확도와 테스트 정확도는 각각 무엇을 의미하나요?
- 각각 학습했던 데이터와 새로운 데이터에 대한 예측 정확도. 정답을 맞춘 비율.
3. 학습 정확도와 테스트 정확도에 차이가 발생하는 이유는 무엇인가요?
- 위의 자료는 과소적합. 학습 데이터와 테스트 데이터 둘 다 정확도가 떨어짐. 모델이 아직 데이터의 특징을 제대로 학습하지 못했음.
- 반대로 과적합인 자료 같은 경우에는 모델이 학습 데이터를 너무 잘 배워서 테스트 데이터에 적응하지 못하는 상황임.
- k 값에 따라서 과적합, 과소적합이 발생할 가능성이 있음. k가 너무 작으면 과적합, 너무 크면 과소적합.

# 문제 6. KNN 성능 평가

5번에서 학습한 KNN의 예측 결과를 평가하세요.

다음 항목을 출력하세요.

1. 혼동행렬 (`confusion_matrix`)
2. 정확도 (`accuracy_score`)
3. 정밀도 (`precision_score`)
4. 재현율 (`recall_score`)
5. F1 점수 (`f1_score`)
6. `classification_report`

이번 데이터는 이진 분류이므로 `0`은 음성, `1`은 양성입니다. 정밀도와 재현율은 양성(1)을 기준으로 계산하세요.

In [68]:
print("혼동행렬\n", confusion_matrix(y_test , pred_test))
print("정확도 : " , accuracy_score(y_test , pred_test))
print("정밀도 : " , precision_score(y_test , pred_test))
print("재현율 : " , recall_score(y_test , pred_test))
print("F1 점수 : " , f1_score(y_test , pred_test))
print(classification_report(y_test , pred_test))

혼동행렬
 [[77 23]
 [27 27]]
정확도 :  0.6753246753246753
정밀도 :  0.54
재현율 :  0.5
F1 점수 :  0.5192307692307693
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       100
           1       0.54      0.50      0.52        54

    accuracy                           0.68       154
   macro avg       0.64      0.64      0.64       154
weighted avg       0.67      0.68      0.67       154



## 분석 질문

1. 실제 양성인데 음성으로 잘못 예측한 데이터(FN)는 몇 개입니까?
- 27개
2. 실제 음성인데 양성으로 잘못 예측한 데이터(FP)는 몇 개입니까?
- 23개
3. 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있나요?
- 정확도는 전체 데이터에서 맞게 예측한 것이고 재현율은 양성 데이터 중 맞게 예측한 것임.
- 고로 정확도와 재현율이 의미하는 바가 다르기 때문에 발생할 수 있음.
4. 당뇨병 양성을 놓치는 것을 줄이려면 어떤 평가 지표를 중요하게 봐야 할까요?
- 재현율을 중요하게 봐야함.

# 문제 7. k값에 따른 성능 비교

KNN의 k값을 변경하면서 학습 정확도와 테스트 정확도가 어떻게 변하는지 확인하세요.

실험할 k값:

`1, 3, 5, 7, 10, 15, 20, 30`

**조건**

- 반복문을 사용하세요.
- 스케일링하지 않은 데이터를 사용하세요.
- 4번에서 결측치를 처리한 데이터를 사용하세요.
- k값 외에는 동일한 조건을 유지하세요.

각 k값의 학습 정확도와 테스트 정확도를 출력하고 다음 표를 완성하세요.

마크다운 또는 코드셀에 주석으로 적어주시면 됩니다.

In [ ]:
def knn(x):
    model = KNeighborsClassifier(n_neighbors=x)
    model.fit(X_train_cleaned , y_train)
    pred_tn = model.predict(X_train_cleaned)
    pred_tt = model.predict(X_test_cleaned)
    acc_tn = accuracy_score(y_train , pred_tn)
    acc_tt = accuracy_score(y_test , pred_tt)
    return acc_tn , acc_tt

list = [1,3,5,7,10,15,20,30]
k_data = []
for i in list:
    a, b = knn(i)
    k_data.append({
        "index" : i ,
        "학습 정확도" : a ,
        "테스트 정확도" : b
    })

st = pd.DataFrame(k_data).set_index("index")
st

,학습 정확도,테스트 정확도
index,,
1,1.000000,0.681818
3,0.842020,0.681818
5,0.801303,0.675325
7,0.785016,0.675325
10,0.798046,0.668831
15,0.791531,0.655844
20,0.788274,0.720779
30,0.765472,0.720779


## 분석 질문

1. 학습 정확도가 가장 높은 k값은 무엇인가요?
- 1
2. 테스트 정확도가 가장 높은 k값은 무엇인가요?
- 20, 30?
3. k가 작을 때 과대적합이 발생하기 쉬운 이유는 무엇인가요?
- 가장 가까운 데이터의 갯수 이므로 너무 적은 데이터만 보고 판단하기 때문.
4. k가 지나치게 커지면 과소적합이 발생할 수 있는 이유는 무엇인가요?
- 너무 넓은 범위의 데이터값을 고려하기 때문에
5. 학습 정확도만 보고 k값을 선택해도 될까요?
- 아 당연히 안됩니다. 학습 정확도와 테스트 정확도 둘다 고려해야함.

# 문제 8. StandardScaler 적용

이번에는 `StandardScaler`로 특성을 표준화한 뒤 동일한 KNN을 학습하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `StandardScaler` 사용
- 스케일러는 학습 데이터에만 `fit()`
- 학습 데이터와 테스트 데이터는 같은 스케일러로 변환
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `StandardScaler` 객체를 생성하세요.
2. 학습 데이터로 `fit()` 하세요.
3. 학습 데이터와 테스트 데이터를 각각 `transform()` 하세요.
4. 변환된 학습 데이터를 DataFrame으로 만들고 `describe()`를 출력하세요.
5. 변환된 데이터로 KNN(k=5)을 학습하세요.
6. 학습 정확도와 테스트 정확도를 출력하세요.
7. 재현율을 출력하세요.

In [70]:
scaler = StandardScaler()

scaler.fit(X_train_cleaned)

X_train_scaled = scaler.transform(X_train_cleaned)
X_test_scaled = scaler.transform(X_test_cleaned)

X_tn_scaled_df = pd.DataFrame(X_train_scaled , columns=X.columns)
X_tt_scaled_df = pd.DataFrame(X_test_scaled , columns=X.columns)

In [71]:
X_tn_scaled_df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02,6.140000e+02
mean,-6.943414e-17,-1.099374e-16,3.095606e-16,-3.471707e-17,-4.339634e-18,1.649061e-16,-1.099374e-16,-1.084908e-16
std,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00,1.000815e+00
min,-1.153338e+00,-2.190541e+00,-3.924957e+00,-2.480958e+00,-1.559139e+00,-2.089621e+00,-1.192094e+00,-1.045895e+00
25%,-8.513551e-01,-7.562208e-01,-6.636760e-01,-4.549828e-01,-2.249692e-01,-7.073644e-01,-7.042617e-01,-7.921693e-01
50%,-2.473897e-01,-1.558076e-01,-1.141979e-02,-4.766137e-03,-1.614373e-01,-7.070181e-03,-2.876344e-01,-3.692933e-01
75%,6.585583e-01,6.113872e-01,6.408364e-01,3.328964e-01,-9.790537e-02,5.942296e-01,4.903223e-01,6.456090e-01
max,3.980368e+00,2.579408e+00,4.065181e+00,7.874025e+00,7.703814e+00,5.081979e+00,5.610293e+00,4.028617e+00


In [72]:
X_tt_scaled_df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000
mean,0.038906,-0.002455,0.100290,0.036894,0.187988,0.006167,-0.083895,-0.052960
std,1.084586,1.074548,0.924732,0.945552,1.413523,1.040927,1.016773,0.971589
min,-1.153338,-2.590817,-1.968188,-2.143295,-1.571846,-2.089621,-1.210275,-1.045895
25%,-0.851355,-0.722865,-0.500612,-0.454983,-0.161437,-0.857689,-0.728502,-0.876744
50%,-0.247390,-0.139129,-0.011420,-0.004766,-0.161437,-0.065734,-0.437620,-0.453869
75%,0.658558,0.661422,0.640836,0.332896,-0.161437,0.608895,0.319884,0.539890
max,3.376402,2.512696,2.760669,3.822075,8.999865,3.952709,5.886024,2.929139


In [73]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_tn_scaled_df , y_train)

pred_tn = model.predict(X_tn_scaled_df)
pred_tt = model.predict(X_tt_scaled_df)

accuracy_train_2 = accuracy_score(y_train , pred_tn)
accuracy_test_2 = accuracy_score(y_test , pred_tt)
print("학습 정확도 : " , accuracy_train_2)
print("테스트 정확도 : " , accuracy_test_2)

recall_2 = recall_score(y_test ,pred_tt)
print("재현율 : " , recall_2)


학습 정확도 :  0.8289902280130294
테스트 정확도 :  0.7532467532467533
재현율 :  0.6111111111111112


## 분석 질문

1. 표준화 후 각 특성의 평균과 표준편차는 대략 얼마인가요?
- 평균은 대략 0 언저리. 표준편차는 1 언저리임.
2. 5번에서 구한 스케일링 전 정확도와 비교하면 어떤 차이가 있나요?
- 스케일링 후 정확도가 더 높음.
3. 모델의 알고리즘과 k값은 그대로인데 정확도가 달라지는 이유는 무엇인가요?
- 스케일링으로 데이터값의 숫자 범위를 균일하게 맞췄기 때문에 가까운 데이터값도 달라짐.

# 문제 9. MinMaxScaler 적용 및 비교

이번에는 `MinMaxScaler`를 적용하고 KNN의 성능을 확인하세요.

**조건**

- 4번에서 결측치를 처리한 데이터 사용
- `MinMaxScaler` 사용
- 학습 데이터에만 `fit()`
- KNN의 k값은 5로 고정

다음 작업을 수행하세요.

1. `MinMaxScaler`로 학습·테스트 데이터를 변환하세요.
2. 변환된 학습 데이터의 `describe()`를 출력하세요.
3. 변환된 데이터로 KNN(k=5)을 학습하세요.
4. 학습 정확도와 테스트 정확도를 출력하세요.
5. 재현율을 출력하세요.
6. 세 가지 전처리 방법의 성능을 비교하세요.

In [74]:
minmax = MinMaxScaler()
minmax.fit(X_train_cleaned)
X_tn_mm = minmax.transform(X_train_cleaned)
X_tt_mm = minmax.transform(X_test_cleaned)
X_tn_mm_df = pd.DataFrame(X_tn_mm , columns=X.columns)
X_tt_mm_df = pd.DataFrame(X_tt_mm , columns=X.columns)

In [75]:
X_tn_mm_df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.224660,0.459238,0.491225,0.239591,0.168320,0.291374,0.175246,0.206107
std,0.194950,0.209817,0.125256,0.096651,0.108045,0.139553,0.147127,0.197224
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.058824,0.300699,0.408163,0.195652,0.144033,0.192740,0.071715,0.050000
50%,0.176471,0.426573,0.489796,0.239130,0.150892,0.290389,0.132962,0.133333
75%,0.352941,0.587413,0.571429,0.271739,0.157750,0.374233,0.247327,0.333333
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [76]:
X_tt_mm_df.describe()

,preg,plas,pres,skin,insu,mass,pedi,age
count,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000
mean,0.232238,0.458723,0.503777,0.243154,0.188615,0.292234,0.162913,0.195671
std,0.211268,0.225275,0.115734,0.091314,0.152600,0.145146,0.149473,0.191465
min,0.000000,-0.083916,0.244898,0.032609,-0.001372,0.000000,-0.002673,0.000000
25%,0.058824,0.307692,0.428571,0.195652,0.150892,0.171779,0.068151,0.033333
50%,0.176471,0.430070,0.489796,0.239130,0.150892,0.282209,0.110913,0.116667
75%,0.352941,0.597902,0.571429,0.271739,0.150892,0.376278,0.222272,0.312500
max,0.882353,0.986014,0.836735,0.608696,1.139918,0.842536,1.040535,0.783333


In [77]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_tn_mm_df , y_train)

pred_tn = model.predict(X_tn_mm_df)
pred_tt = model.predict(X_tt_mm_df)

accuracy_train_3 = accuracy_score(y_train , pred_tn)
accuracy_test_3 = accuracy_score(y_test , pred_tt)
print("학습 정확도 : " , accuracy_train_3)
print("테스트 정확도 : " , accuracy_test_3)

recall_3 = recall_score(y_test , pred_tt)
print("재현율 : " , recall_3)

학습 정확도 :  0.8322475570032574
테스트 정확도 :  0.7337662337662337
재현율 :  0.5555555555555556


In [78]:
compare_df = pd.DataFrame({
    "index": ["스케일링 없음", "StandardScaler", "MinMaxScaler"],
    "학습 정확도": [accuracy_train_1, accuracy_train_2, accuracy_train_3],
    "테스트 정확도": [accuracy_test_1, accuracy_test_2, accuracy_test_3],
    "재현율": [recall_1, recall_2, recall_3],
}).set_index("index")

compare_df

,학습 정확도,테스트 정확도,재현율
index,,,
스케일링 없음,0.801303,0.675325,0.500000
StandardScaler,0.828990,0.753247,0.611111
MinMaxScaler,0.832248,0.733766,0.555556


## 분석 질문

1. 테스트 정확도가 가장 높은 전처리 방법은 무엇인가요?
- StandardScaler
2. 재현율이 가장 높은 전처리 방법은 무엇인가요?
- StandardScaler
3. 정확도가 높은 방법과 재현율이 높은 방법이 다를 수도 있나요?
-  다를 수도 있쥬.
4. 스케일링을 적용한다고 해서 정확도가 반드시 높아지는 것은 아닌 이유는 무엇일까요?
- 스케일링은 value들의 숫자 범위를 균일하게 맞추는 전처리 작업임. 상관관계를 고려하지 않은 작업이기 때문에 정확도가 반드시 높아진다고 할 수 없음.

# 문제 10. 종합 — KNN 설정 선택 및 최종 분석

지금까지 실험한 결과를 이용하여 가장 적절하다고 생각하는 KNN 설정을 선택하세요.

## 1단계 — 스케일링 방법 선택**

다음 중 하나를 선택하세요.

- 스케일링 없음
- StandardScaler
- MinMaxScaler

In [79]:
print("선택한 스케일링 방법 : StandardScaler")

scaler = StandardScaler()
scaler.fit(X_train_cleaned)
X_tn_scaled_df = pd.DataFrame(scaler.transform(X_train_cleaned) , columns=X.columns)
X_tt_scaled_df = pd.DataFrame(scaler.transform(X_test_cleaned) , columns=X.columns)

선택한 스케일링 방법 : StandardScaler


## 2단계 — k값 실험

선택한 스케일링 방법으로 다음 k값을 실험하세요.

`1, 3, 5, 7, 10, 15, 20, 30`

각 k값에 대해 다음 결과를 표로 정리하세요.

- 학습 정확도
- 테스트 정확도
- 재현율

In [80]:
list = [1,3,5,7,10,15,20,30]
data = []

def scaling(x):
    model = KNeighborsClassifier(n_neighbors=x)
    model.fit(X_tn_scaled_df , y_train)
    pred_tn = model.predict(X_tn_scaled_df)
    pred_tt = model.predict(X_tt_scaled_df)
    accuracy_train_4 = accuracy_score(y_train , pred_tn)
    accuracy_test_4 = accuracy_score(y_test , pred_tt)
    recall_4 = recall_score(y_test , pred_tt)
    return accuracy_train_4 , accuracy_test_4 , recall_4

for i in list:
    a , b , c = scaling(i)
    data.append({
        "k" : i ,
        "학습 정확도" : a ,
        "테스트 정확도" : b ,
        "재현율" : c
    })

data_df = pd.DataFrame(data).set_index("k")
data_df

,학습 정확도,테스트 정확도,재현율
k,,,
1,1.000000,0.720779,0.555556
3,0.846906,0.720779,0.611111
5,0.828990,0.753247,0.611111
7,0.817590,0.727273,0.555556
10,0.804560,0.740260,0.537037
15,0.802932,0.740260,0.555556
20,0.788274,0.759740,0.555556
30,0.788274,0.740260,0.518519


## 3단계 — 최종 설정 선택

실험 결과를 비교하여 다음 두 가지를 결정하세요.

1. 사용할 스케일링 방법
2. 사용할 k값

단순히 정확도가 가장 높은 설정을 선택해도 되지만, 재현율이나 과대적합·과소적합 가능성을 고려하여 다른 설정을 선택해도 됩니다.

In [81]:
print("사용할 스케일링 방법 : MinMaxScaler")
print("사용할 k 값 : 5")

minmax = MinMaxScaler()
minmax.fit(X_train_cleaned)
X_tn_mm = minmax.transform(X_train_cleaned)
X_tt_mm = minmax.transform(X_test_cleaned)

사용할 스케일링 방법 : MinMaxScaler
사용할 k 값 : 5


## 4단계 — 선택한 설정으로 KNN 학습 및 평가

선택한 설정으로 KNN을 학습하고 다음 결과를 출력하세요.

1. 학습 정확도
2. 테스트 정확도
3. 혼동행렬
4. 정밀도
5. 재현율
6. F1 점수

In [82]:
model=KNeighborsClassifier(n_neighbors=5)
model.fit(X_tn_mm , y_train)
pred_train = model.predict(X_tn_mm)
pred_test = model.predict(X_tt_mm)
print("1. 학습 정확도 : " , accuracy_score(y_train , pred_train))
print("2. 테스트 정확도 : " , accuracy_score(y_test , pred_test))
print("3. 혼동행렬 : \n" , confusion_matrix(y_test , pred_test))
print("4. 정밀도 : " , precision_score(y_test , pred_test))
print("5. 재현율 : " , recall_score(y_test , pred_test))
print("6. F1점수 : " , f1_score(y_test , pred_test))

1. 학습 정확도 :  0.8322475570032574
2. 테스트 정확도 :  0.7337662337662337
3. 혼동행렬 : 
 [[83 17]
 [24 30]]
4. 정밀도 :  0.6382978723404256
5. 재현율 :  0.5555555555555556
6. F1점수 :  0.594059405940594


## 5단계 — 최종 분석 보고서

마크다운 셀에 다음 질문의 답변을 작성하세요.

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?
- MinmaxScaler 와 k=5 를 선택했음.
2. 해당 설정을 선택한 이유는 무엇인가요?
- 바로 위에 StandardScaler 와 비교하고 싶었음.
3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?
- 스케일링 안한것보단 한게 더 정확했음.
4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?
- k값이 너무 작으면 과적합, 너무 크면 과소적합 현상이 보임. 적당한게 제일 좋음.
5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이유를 설명하세요.
- 재현율을 더 중요하게 고려했음. 재현율은 실제 양성값 중 제대로 예측한 비율임.
- 양성값을 찾아내는게 주 목적이라고 생각하기 때문.
6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?
- 그건 왜일까요. 진짜 알수가 없네. 학습은 학습데이터로 할거고 거기에 fit 할거라서 맞출거라서 그런가.
7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?
- 데이터 전처리 단계가 제일 중요한 거 같음. 애초에 머신에 넣을 데이터 자체가 제대로 되지 못하다면 그 과정이 어떻든 간에 잘못된 결과값이 나올 수 밖에 없기 때문.


**참고:** 이번 과제는 아직 배우지 않은 검증 데이터·교차검증을 사용하지 않고, 동일한 테스트 데이터로 여러 설정의 결과를 비교합니다. 이는 수업 내용을 복습하기 위한 방식입니다. 실제 프로젝트에서는 설정 선택에 사용한 데이터와 최종 성능 평가에 사용할 테스트 데이터를 별도로 구분해야 합니다.